In [ ]:
!pip install -q bitsandbytes
!pip install -q lm-eval

In [ ]:
import torch
from torch.optim import AdamW
from transformers import (AutoModelForCausalLM, Gemma3ForConditionalGeneration, AutoProcessor,
                          AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments)
from tqdm import tqdm
from datasets import load_dataset
from torch.nn import functional as F
from torch.utils.data import DataLoader

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] =  userdata.get('HF_TOKEN')
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
teacher_model_name = "meta-llama/Llama-3.2-3B"
student_model_name = "meta-llama/Llama-3.2-1B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(student_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'<|end_of_text|>'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch_dtype = torch.bfloat16
attn_implementation = "eager"

In [ ]:
teacher_model = AutoModelForCausalLM.from_pretrained(teacher_model_name,
                                                     device_map="auto")

student_model = AutoModelForCausalLM.from_pretrained(student_model_name,
                                                     device_map="auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
dataset = load_dataset("lavita/medical-qa-datasets", "all-processed", split="train")
original_dataset = dataset
dataset = dataset.select(range(1000))

In [ ]:
dataset.column_names

['instruction', 'input', 'output', '__index_level_0__']

In [ ]:
def format_prompt(example):
    return {
        "prompt": f"Instruction: {example['instruction']}\nQuestion{example['input']}\nAnswer: {example['output']}"
    }

formatted_dataset = dataset.map(format_prompt)

In [ ]:
print(formatted_dataset['prompt'][0])

Instruction: If you are a doctor, please answer the medical questions based on the patient's description.
Questionhi. im a home health aide and i have a client with scoliosis in the back and kidney disease. her feet ankles and calves have been swollen for the past 2 weeks. mostly in her feet. she started a patch for pain in her legs 3 weeks ago. she started swelling up almost a week after she started the patch and the pain doctor cut the dose in half and she is still swollen. she has no blood clots in her legs because one of her doctors checked and they said it might be because of her back. what do you think? im concerned because she has been swollen up for to long and theres only so much i can do being her home health aide. we both want to get to the bottom of this swelling she is having
Answer: hi, thanks for contacting chatbot. swelling in the legs and feet can come from many causes, one of them being general circulation or ineffectiveness of the kidneys to rid the body of excess wa

In [ ]:
def tokenize(samples):
    tokenized = tokenizer(samples["prompt"],
                          padding="max_length",
                          truncation=True,
                          max_length=128,
                          return_tensors="pt")

    input_ids = tokenized["input_ids"]
    labels = input_ids.clone()
    attention_mask = tokenized["attention_mask"]

    return{
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }

In [ ]:
print("Tokenizing dataset .... ")
tokenized_dataset = formatted_dataset.map(tokenize,
                                          batched=True,
                                          batch_size=32,
                                          remove_columns=dataset.column_names,
                                          desc="Processing samples",
                                          load_from_cache_file=False)

Tokenizing dataset .... 


Processing samples:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset.set_format("torch")

In [ ]:
dataloader = DataLoader(
    tokenized_dataset,
    batch_size=4,
    shuffle=True
)

### Optimizer

In [ ]:
lr = 1e-5
optimizer = AdamW(student_model.parameters(), lr=lr)

### Training Loop

In [ ]:
num_epochs = 10
temperature = 2.0
alpha = 1
accumulation_steps = 8

In [ ]:
for epoch in range(num_epochs):
    student_model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.no_grad():
            teacher_outputs = teacher_model(
                input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            teacher_logits = teacher_outputs.logits / temperature

        student_outputs = student_model(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        student_logits = student_outputs.logits

        teacher_probs = F.softmax(teacher_logits, dim=-1)
        student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
        loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
        loss = loss / accumulation_steps

        loss.backward()

        if ((batch_idx + 1) % accumulation_steps == 0) or (batch_idx + 1 == len(dataloader)):
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")

### Store the Model

In [ ]:
distil_student_model_name = "pruned_distil_Llama-3.2-1B"

In [ ]:
student_model.save_pretrained(distil_student_model_name)
tokenizer.save_pretrained(distil_student_model_name)

In [ ]:
student_model.push_to_hub(distil_student_model_name,
                          private=False,
                          use_temp_dir=False)
tokenizer.push_to_hub(distil_student_model_name,
                      private=False,
                      use_temp_dir=False)

### Evaluating the model

In [ ]:
from lm_eval import evaluator, tasks, models

def evaluate_hf_model(model_name, tasks=['arc_easy'], num_fewshot=0):
    model_args = f"pretrained={model_name},device=cuda"
    tasks = tasks

    results = evaluator.simple_evaluate(
      model="hf",
      model_args=model_args,
      tasks=tasks,
      num_fewshot=0,
      limit=None,
      bootstrap_iters=10
    )

    metrics = results.get('results', {})
    return metrics

In [ ]:
tasks = ['lambada']

metrics_pruned_kd = evaluate_hf_model("TachyHealthResearch/pruned_distil_Llama-3.2-1B", tasks=tasks)

metrics_pruned_kd